In [3]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [9]:
from operator import itemgetter
from typing import Dict, List
from langchain_community.tools.tavily_search import TavilySearchResults

from langchain_core.messages import AIMessage
from langchain_core.runnables import Runnable, RunnablePassthrough
from langchain_core.tools import tool
from langchain.tools import Tool
from langchain.utilities import SerpAPIWrapper
from pydantic import BaseModel
from enum import Enum

class TicketType(str, Enum):
    bug = "bug"
    feature_request = "feature_request"
    other = "other"

class TicketPriority(str, Enum):
    low = "low"
    medium = "medium"
    high = "high"

class ProductDetails(BaseModel):
    product_name: str

class Ticket(BaseModel):
    ticket_type: TicketType
    ticket_description: str
    ticket_title: str
    ticket_priority: TicketPriority

@tool
def search_knowledge_base_articles(subject:str) -> str:
    """search internal documentation given a subject"""
    articles = ["How to troubleshoot network issues", "How to reset your password", "How to update your email address"]
    return f"Here are some articles on {subject} from our knowledge base: {articles}"

@tool
def create_ticket(ticket:Ticket) -> str:
    """create a ticket in the database"""
    return f"Successfully created a {ticket.ticket_type} ticket with title {ticket.ticket_title} and description {ticket.ticket_description}."

@tool
def search_products(product_name: str) -> List[str]:
    """Search for similar products in the database"""
    return [f"MAcbook Pro", f"Macbook Air", f"Macbook Pro 2021"]


@tool
def order_product(product_name:str) -> str:
    """Order a product from the database using the product details"""
    if product_name not in ["Macbook Pro", "Macbook Air", "Macbook Pro 2021"]:
        return f"Product {product_name} not found in the database"
    return f"Successfully ordered {product_name}"


tools = [search_products, order_product,create_ticket, search_knowledge_base_articles]
llm_with_tools = llm.bind_tools(tools)


def call_tools(msg: AIMessage) -> List[Dict]:
    """Simple sequential tool calling helper."""
    tool_map = {tool.name: tool for tool in tools}
    tool_calls = msg.tool_calls.copy()
    for tool_call in tool_calls:
        tool_call["output"] = tool_map[tool_call["name"]].invoke(tool_call["args"])
    return tool_calls




In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.messages import HumanMessage
# Adapted from https://smith.langchain.com/hub/hwchase17/openai-tools-agent
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. You may not need to use tools for every query - the user may just want to chat!",
        ),
        MessagesPlaceholder(variable_name="messages"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [15]:


agent = create_openai_tools_agent(llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [16]:
output = agent_executor.invoke({"messages": [HumanMessage(content="do we have any Mackbook in out product list?")], "agent_scratchpad": {}})



> Entering new AgentExecutor chain...

Invoking: `search_products` with `{'product_name': 'Macbook'}`


['MAcbook Pro', 'Macbook Air', 'Macbook Pro 2021']Yes, we have the following Mackbook products in our product list:
1. Macbook Pro
2. Macbook Air
3. Macbook Pro 2021

Is there a specific model you are interested in?

> Finished chain.


In [17]:
flow_example = """
do we have any Mackbook in out product list?
I want to order a MacBook Pro 2021
I want to create a bug ticket for the product search tool is very slow the problem is critical and needs to be fixed asap
How to troubleshoot network issues?
"""


In [18]:
messages = []
while True:
    user_input = input("Enter your message: q to quit: ")
    if user_input == "q":
        break
    messages.append(HumanMessage(content=user_input))
    output = agent_executor.invoke({"messages": messages})
    print(output['output'])
    messages.append(AIMessage(content=output['output']))







> Entering new AgentExecutor chain...

Invoking: `search_knowledge_base_articles` with `{'subject': 'network troubleshooting'}`


Here are some articles on network troubleshooting from our knowledge base: ['How to troubleshoot network issues', 'How to reset your password', 'How to update your email address']I found some articles on network troubleshooting in our knowledge base. Would you like to know more about them?

> Finished chain.
I found some articles on network troubleshooting in our knowledge base. Would you like to know more about them?
